In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')

print(invoices.shape)
print(payments.shape)

(18033, 10)
(19257, 5)


In [5]:
print(invoices['invoice_id'].isnull().sum())
print(payments['payment_id'].isnull().sum())

0
0


In [6]:
# Method A
dup_rows_A = invoices[invoices.duplicated(subset='invoice_id', keep=False)]
print("Method A - affected rows:", len(dup_rows_A))

# Method B
group_sizes = invoices.groupby('invoice_id').size()
dup_groups = group_sizes[group_sizes > 1]
print("Method B - duplicate groups:", len(dup_groups))
print("Method B - affected rows:", dup_groups.sum())

Method A - affected rows: 393
Method B - duplicate groups: 196
Method B - affected rows: 393


In [7]:
# Method A
dup_rows_A_pay = payments[payments.duplicated(subset='payment_id', keep=False)]
print("Method A - affected rows:", len(dup_rows_A_pay))

# Method B
group_sizes_pay = payments.groupby('payment_id').size()
dup_groups_pay = group_sizes_pay[group_sizes_pay > 1]
print("Method B - duplicate groups:", len(dup_groups_pay))
print("Method B - affected rows:", dup_groups_pay.sum())

Method A - affected rows: 403
Method B - duplicate groups: 201
Method B - affected rows: 403


In [8]:
# Invoices
print("Invoice group size distribution:")
print(dup_groups.value_counts())

# Payments
print("\nPayment group size distribution:")
print(dup_groups_pay.value_counts())

Invoice group size distribution:
2    195
3      1
Name: count, dtype: int64

Payment group size distribution:
2    200
3      1
Name: count, dtype: int64


In [9]:
# Invoices - show all duplicate rows, grouped and sorted for easy comparison
invoice_dupes_sorted = invoices[invoices.duplicated(subset='invoice_id', keep=False)].sort_values('invoice_id')
invoice_dupes_sorted.head(20)

,invoice_id,so_id,customer_id,branch_id,invoice_date,due_date,total_order_value,total_gst_amount,grand_total,payment_status
14917,INV-105711,SO-901766,C0208,KOL001,2021-11-20,2022-01-04,1424270,368413.6,1792683.6,Paid
8324,INV-105711,SO-549003,C0191,KOL001,2024-12-15,2025-01-14,529940,146609.2,676549.2,Paid
14643,INV-106596,SO-807417,C0281,KOL001,2024-01-25,2024-03-25,40100,7218.0,47318.0,Paid
9894,INV-106596,SO-399413,C0409,PUN001,2021-06-16,2021-07-31,1204730,336001.4,1540731.4,Paid
1712,INV-111054,SO-233671,C0072,PUN001,2023-04-11,2023-04-26,1207780,319950.4,1527730.4,Paid
1706,INV-111054,SO-316263,C0313,AHM001,2019-08-02,2019-10-01,310,55.8,365.8,Paid
13853,INV-112235,SO-236576,C0220,KOL001,2021-04-17,2021-06-16,1479960,405362.8,1885322.8,Paid
946,INV-112235,SO-363636,C0198,AHM001,2021-01-08,2021-03-09,2369250,654525.0,3023775.0,Paid
13492,INV-112378,SO-911241,C0082,AHM001,2025-01-01,2025-03-02,3833070,1071302.6,4904372.6,Paid
1609,INV-112378,SO-870768,C0104,KOL001,2019-04-15,2019-06-14,2036600,570248.0,2606848.0,Paid


In [10]:
payment_dupes_sorted = payments[payments.duplicated(subset='payment_id', keep=False)].sort_values('payment_id')
payment_dupes_sorted.head(20)

,payment_id,invoice_id,payment_date,payment_amount,payment_method
19220,PAY-102690,INV-826669,2021-04-23,3837051.40,Credit Card
7121,PAY-102690,INV-196859,2021-12-29,520518.80,Cash
16981,PAY-105592,INV-963449,2023-02-23,2123180.75,Cash
11803,PAY-105592,INV-323176,2022-07-10,1019902.14,Cheque
15115,PAY-112476,INV-743633,2024-06-17,4544341.00,Credit Card
5761,PAY-112476,INV-374653,2024-05-16,1724913.40,Cash
1148,PAY-115474,INV-645256,2022-12-23,473838.80,Credit Card
3929,PAY-115474,INV-583030,2023-08-25,3764707.00,Credit Card
5117,PAY-115921,INV-301855,2022-01-06,353798.70,UPI
9009,PAY-115921,INV-121000,2021-05-08,246614.08,Cash


In [11]:
# Invoices - which branches are affected?
print(invoice_dupes_sorted['branch_id'].value_counts())

# Payments - which payment methods are affected?
print(payment_dupes_sorted['payment_method'].value_counts())

branch_id
CHN001    85
KOL001    74
PUN001    71
AHM001    67
DEL001    53
HYD001    43
Name: count, dtype: int64
payment_method
Credit Card      90
Cheque           90
Cash             80
UPI              72
Bank Transfer    71
Name: count, dtype: int64


In [12]:
# Invoices - extract year from invoice_date, count affected rows per year
invoice_dupes_sorted['invoice_date'] = pd.to_datetime(invoice_dupes_sorted['invoice_date'])
print(invoice_dupes_sorted['invoice_date'].dt.year.value_counts().sort_index())

# Payments - extract year from payment_date, count affected rows per year
payment_dupes_sorted['payment_date'] = pd.to_datetime(payment_dupes_sorted['payment_date'])
print(payment_dupes_sorted['payment_date'].dt.year.value_counts().sort_index())

invoice_date
2019    64
2020    61
2021    74
2022    63
2023    60
2024    68
2025     3
Name: count, dtype: int64
payment_date
2019    51
2020    62
2021    67
2022    70
2023    70
2024    77
2025     6
Name: count, dtype: int64


In [13]:
print(invoices[invoices['invoice_id'] == 'INV-341709'])
print(payments[payments['payment_id'] == 'PAY-361416'])

       invoice_id      so_id customer_id branch_id invoice_date    due_date  \
10098  INV-341709  SO-533236       C0345    PUN001   2021-07-10  2021-08-09   
14416  INV-341709  SO-510296       C0176    CHN001   2024-08-11  2024-08-26   
14873  INV-341709  SO-455910       C0078    HYD001   2022-01-18  2022-02-17   

       total_order_value  total_gst_amount  grand_total payment_status  
10098             803850          222313.0    1026163.0           Paid  
14416             551950          136641.0     688591.0           Paid  
14873            2281250          606975.0    2888225.0           Paid  
       payment_id  invoice_id payment_date  payment_amount payment_method
1338   PAY-361416  INV-781845   2019-05-12      1732917.00  Bank Transfer
1995   PAY-361416  INV-625724   2023-10-18      1111533.42    Credit Card
12071  PAY-361416  INV-652064   2024-02-01       416278.75    Credit Card


In [14]:
print(payments.columns.tolist())

['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']


In [15]:
print(invoice_dupes_sorted['payment_status'].value_counts(normalize=True))
print(invoices['payment_status'].value_counts(normalize=True))

payment_status
Paid              0.692112
Partially Paid    0.208651
Unpaid            0.099237
Name: proportion, dtype: float64
payment_status
Paid              0.695170
Partially Paid    0.202961
Unpaid            0.101869
Name: proportion, dtype: float64
